## Week 2: Attribution Logic
In this section, we analyze customer journeys using event-level data and build simple attribution models such as first-touch, last-touch, and linear attribution.

In [4]:
import pandas as pd

events = pd.read_csv(
    "events.csv",
    usecols=["customer_id", "timestamp", "campaign_id", "event_type", "traffic_source"]
)

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
events.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
0,2021-01-14 13:35:43,43812,view,Email,43.0
1,2021-12-03 21:36:50,71340,add_to_cart,Email,10.0
2,2021-12-27 08:25:15,59540,purchase,Organic,0.0
3,2022-01-22 15:06:54,3601,add_to_cart,Paid Search,30.0
4,2021-05-10 12:03:09,92735,bounce,Email,26.0


### Step 1: Keep only valid journey records
We keep only rows with usable timestamps and sort the event stream customer by customer.

In [6]:
events_clean = events.dropna(subset=["timestamp", "customer_id"]).copy()
events_clean = events_clean.sort_values(["customer_id", "timestamp"])
events_clean.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
335888,2021-06-13 12:35:27,1,view,Organic,0.0
473478,2021-07-09 19:19:25,1,add_to_cart,Paid Search,29.0
428621,2021-11-28 20:19:24,1,view,EMAIL,35.0
94193,2021-12-07 07:04:24,1,click,Paid Search,47.0
572035,2021-12-16 14:19:02,1,view,Paid Search,28.0


### Step 2: Identify conversion events
We treat purchase events as conversions and use them to connect earlier touchpoints to final outcomes.

In [8]:
purchase_events = events_clean[events_clean["event_type"].str.lower() == "purchase"].copy()
purchase_events.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id
219621,2023-11-19 07:55:42,8,purchase,Paid Search,26.0
470867,2022-08-31 08:52:51,13,purchase,Paid Search,50.0
79339,2022-06-22 12:48:33,14,purchase,Organic,0.0
699187,2022-11-05 02:11:12,22,purchase,Social,40.0
544555,2022-05-06 23:28:54,23,purchase,Paid Search,42.0


### Step 3: Build pre-conversion touchpoints
For each customer who converted, we collect all events that happened before or at the purchase time.

In [10]:
journey_data = events_clean.merge(
    purchase_events[["customer_id", "timestamp"]].rename(columns={"timestamp": "purchase_time"}),
    on="customer_id",
    how="inner"
)

journey_data = journey_data[journey_data["timestamp"] <= journey_data["purchase_time"]].copy()
journey_data.head()

,timestamp,customer_id,event_type,traffic_source,campaign_id,purchase_time
0,2021-10-07 07:04:43,8,view,Social,3.0,2023-11-19 07:55:42
1,2022-03-01 09:30:22,8,view,Social,18.0,2023-11-19 07:55:42
2,2023-05-30 22:27:35,8,click,Social,24.0,2023-11-19 07:55:42
3,2023-11-02 13:19:38,8,view,Organic,0.0,2023-11-19 07:55:42
4,2023-11-19 07:55:42,8,purchase,Paid Search,26.0,2023-11-19 07:55:42


### Step 4: First-touch attribution
First-touch attribution gives full credit to the first recorded campaign touchpoint in the conversion journey.

In [12]:
first_touch = journey_data.sort_values(["customer_id", "purchase_time", "timestamp"]) \
    .groupby(["customer_id", "purchase_time"], as_index=False).first()

first_touch_summary = first_touch.groupby("campaign_id").size().reset_index(name="first_touch_conversions")
first_touch_summary = first_touch_summary.sort_values("first_touch_conversions", ascending=False)
first_touch_summary.head(10)

,campaign_id,first_touch_conversions
0,0.0,18651
44,44.0,483
31,31.0,478
17,17.0,474
45,45.0,474
4,4.0,468
18,18.0,463
29,29.0,462
19,19.0,460
32,32.0,452


### Step 5: Last-touch attribution
Last-touch attribution gives full credit to the final campaign touchpoint before conversion.

In [14]:
last_touch = journey_data.sort_values(["customer_id", "purchase_time", "timestamp"]) \
    .groupby(["customer_id", "purchase_time"], as_index=False).last()

last_touch_summary = last_touch.groupby("campaign_id").size().reset_index(name="last_touch_conversions")
last_touch_summary = last_touch_summary.sort_values("last_touch_conversions", ascending=False)
last_touch_summary.head(10)

,campaign_id,last_touch_conversions
0,0.0,8083
44,44.0,920
5,5.0,917
29,29.0,890
7,7.0,888
18,18.0,876
49,49.0,857
17,17.0,828
25,25.0,815
8,8.0,799


### Step 6: Linear attribution
Linear attribution splits one conversion equally across all touchpoints in the journey.

In [16]:
journey_counts = journey_data.groupby(["customer_id", "purchase_time"]).size().reset_index(name="num_touches")
linear_data = journey_data.merge(journey_counts, on=["customer_id", "purchase_time"], how="left")
linear_data["linear_credit"] = 1 / linear_data["num_touches"]

linear_summary = linear_data.groupby("campaign_id", as_index=False)["linear_credit"].sum()
linear_summary = linear_summary.sort_values("linear_credit", ascending=False)
linear_summary.head(10)

,campaign_id,linear_credit
0,0.0,16059.734311
44,44.0,609.883588
18,18.0,573.078135
29,29.0,567.462012
17,17.0,560.041790
7,7.0,547.304977
5,5.0,546.400468
49,49.0,546.242135
8,8.0,537.907077
25,25.0,537.308052


### Step 7: Compare attribution models
We compare how campaign performance changes under first-touch, last-touch, and linear attribution.

In [18]:
attribution_compare = first_touch_summary.merge(
    last_touch_summary, on="campaign_id", how="outer"
).merge(
    linear_summary, on="campaign_id", how="outer"
)

attribution_compare = attribution_compare.fillna(0)
attribution_compare = attribution_compare.sort_values("last_touch_conversions", ascending=False)
attribution_compare.head(15)

,campaign_id,first_touch_conversions,last_touch_conversions,linear_credit
0,0.0,18651,8083,16059.734311
44,44.0,483,920,609.883588
5,5.0,428,917,546.400468
29,29.0,462,890,567.462012
7,7.0,450,888,547.304977
18,18.0,463,876,573.078135
49,49.0,436,857,546.242135
17,17.0,474,828,560.041790
25,25.0,441,815,537.308052
8,8.0,443,799,537.907077


### Step 8: Add campaign channel names
To make the output easier to interpret, we attach channel information from the campaigns table.

In [21]:
campaigns = pd.read_csv("campaigns.csv")
campaign_lookup = campaigns[["campaign_id", "channel", "objective"]].drop_duplicates()

attribution_compare_named = attribution_compare.merge(
    campaign_lookup, on="campaign_id", how="left"
)

attribution_compare_named.head(15)

,campaign_id,first_touch_conversions,last_touch_conversions,linear_credit,channel,objective
0,0.0,18651,8083,16059.734311,NaN,NaN
1,44.0,483,920,609.883588,Affiliate,Reactivation
2,5.0,428,917,546.400468,Social,Acquisition
3,29.0,462,890,567.462012,Email,Acquisition
4,7.0,450,888,547.304977,Paid Search,Cross-sell
5,18.0,463,876,573.078135,Affiliate,Retention
6,49.0,436,857,546.242135,Paid Search,Reactivation
7,17.0,474,828,560.041790,Display,Retention
8,25.0,441,815,537.308052,Paid Search,Cross-sell
9,8.0,443,799,537.907077,Paid Search,Cross-sell


### Week 2 observations
- First-touch shows which campaigns introduced customers.
- Last-touch shows which campaigns were closest to conversion.
- Linear attribution shows which campaigns consistently appeared across the full journey.
- These results will support Week 3 KPI calculations such as efficiency and ROI analysis.